In [ ]:
!pip3 install transformers accelerate datasets evaluate scikit-learn

## Load Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset("zivast/KKS-sentiment-workshop")
train_dataset = dataset["train"]
print("Train data size", len(train_dataset))
val_dataset = dataset["validation"]
print("Val data size", len(val_dataset))

train_dataset

## Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "EMBEDDIA/crosloengual-bert"
num_labels = len(set(train_dataset["label"]))
print("Number of labels:", num_labels)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Prepare data

In [ ]:
MAX_LENGTH = 512

def tokenize_example(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_example, num_proc=8)
val_dataset = val_dataset.map(tokenize_example, num_proc=8)
print("First training example:", train_dataset[0])

## Setup Training

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    }

In [ ]:
from transformers import TrainingArguments

# Batch size per GPU
micro_batch_size = 16
# Global batch size
batch_size = 16
# Accumulate the gradients to achieve the batch size
gradient_accumulation_steps = batch_size // micro_batch_size

# Number of training epochs
num_epochs = 3
steps_per_epoch = len(train_dataset) // batch_size
eval_steps = int(1 / 2 * steps_per_epoch)  # Evaluate 2 times per epoch
save_steps = int(1 / 2 * steps_per_epoch)  # Save 2 times per epoch

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Evaluate each {eval_steps} steps ({eval_steps / steps_per_epoch:.2f} epochs)")
print(f"Save each {save_steps} steps ({save_steps / steps_per_epoch:.2f} epochs)")

args = TrainingArguments(
    # Output settings
    output_dir="tmp",  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=num_epochs,
    # Batch size settings
    per_device_train_batch_size=micro_batch_size,
    per_device_eval_batch_size=micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-5,  # Standard fine-tuning LR for BERT
    weight_decay=0.01,
    # Learning rate schedule
    warmup_steps=100,  # Number of steps when LR is linearly increasing
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": 2e-6},
    # Logging and saving
    logging_steps=10,  # Log metrics every N steps
    eval_on_start=True, # Perform first evaluation before starting the training
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # Precision settings
    bf16=True,
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to="none"  # Disable external logging
)

## Train

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

## Save model

In [ ]:
model_output_path = "CroSloEngual-BERT-Sent"
trainer.save_model(model_output_path)
tokenizer.save_pretrained(model_output_path)